<!-- # Consultas 查询板块特征衍生（四单元格版｜逐行注释）

本 Notebook 严格按你要求拆分为 4 个代码格：

- **代码格 1：读取数据**（只做读入 + 基础校验 + 时间字段解析）
- **代码格 2：处理数据**（解析 `response_body`，平铺 `consultas`，机构归类，计算窗口统计所需字段）
- **代码格 3：衍生函数逻辑**（把明细聚合为特征表：多窗口 × 17 类，新增：个征信机构=SIC）
- **代码格 4：生成结果**（调用函数、导出结果到 `outputs/`）

数据源：`衍生原始数据_331条.csv`

输出文件（最终只保留 2 个）：
- `outputs/consultas_flat.csv`：consultas 平铺明细（`apply_id` 会重复）
- `outputs/features_consultas.csv`：按 `apply_id` 聚合后的特征表

> 关于“查询日期的均值/标准差”口径：
> 
> 直接对日期做均值/方差不直观且不同实现差异大，因此这里采用更常见且可解释的数值化口径：
> 
> **`days_before_request = request_time - fechaConsulta`（单位：天）**，并对该数值做均值/标准差。 -->


In [77]:
# 代码格 1/4：读取数据（逐行注释版）  # 单元格标题：说明这是第 1 个代码格，职责是“读入+校验”
# 目标：把 CSV 读成 df_raw，并确保关键字段存在且 request_time 可解析为 datetime  # 单元格的输入/输出与目的概述
#
# ========================= 这套衍生“到底衍生了什么”（总体说明）=========================
# 1) 一共衍生了多少个特征？
#    - 时间窗口：7 个（30/60/90/120/180/360/720 天）
#    - 每个窗口都会生成：
#      (A) 1 个“总查询次数”特征
#      (B) 机构 17 大类 × 7 个统计特征（新增：个征信机构=SIC；在原 5 个统计基础上新增 2 个“每天次数”特征）
#      (C) tipoCredito 三类(CC/PP/TC) × 7 个统计特征（同样新增 2 个“每天次数”特征）
#    - 所以：
#      * 每窗口特征数 = 1 + (17×7) + (3×7) = 141
#      * 总特征数 = 141 × 7 = 987
#    - 另外输出表里还会带上 request_time（这是原始截止时间，不算衍生特征）。
#
# 2) 特征分块（不用记具体列名，只要记“每块算什么”）
#    - [块1：总查询强度]：近 N 天内，总共发生了多少条 consultas 查询记录（不分机构、不分 tipoCredito）。
#    - [块2：按机构大类的查询画像（17 类）]：对每个大类（商店信息/大众金融协会/.../政府/个征信机构），在近 N 天内分别计算：
#      - 数量（cnt）：这个大类查了多少次
#      - 占比（ratio）：这个大类次数 / 全部查询次数
#      - 平均新鲜度（days_mean）：这些查询平均距离 request_time 多少天（越小=越近期）
#      - 新鲜度波动（days_std）：这些查询在时间上分散不分散
#      - 有效值占比（notnull_ratio）：能算出 days_before_request 的记录占比（更多用于数据质量）
#      - 每天平均次数（daily_cnt_mean）：该类别近 N 天内的次数 / N（没发生的天按 0 计入均值，比如 3/30=0.1）
#      - 每天次数标准差（daily_cnt_std）：把近 N 天每天的次数当作序列 x1..xN（没发生的天为0），计算标准差
#    - [块3：按 tipoCredito 的查询画像（三类）]：对 CC/PP/TC 三类，在近 N 天内计算同样 7 个统计。
#
# 3) 特征逻辑（核心口径）
#    - “截止时间”使用每个申请的 request_time。
#    - 每条查询有一个查询日期 fechaConsulta。
#    - 我们把日期转成可统计的数值：
#        days_before_request = request_time - fechaConsulta （单位：天）
#      然后在不同窗口里用条件：0 <= days_before_request <= N 来筛选“近 N 天内”的查询。
#
# 4) 遇到不同数据类型/不同异常时如何处理（你关心的“空值/异常”口径）
#    - JSON 类型（response_body）：
#      * 正常：是合法 JSON 字符串 -> json.loads 解析成 dict，再取 consultas list。
#      * 情况 C（非法/空 JSON）：解析失败或为空 -> 该 apply_id 的所有衍生特征统一置 -999。
#      * 情况 D（没有 consultas 块）：JSON 合法但没有 consultas key（或 consultas 不是 list） -> 该 apply_id 的所有衍生特征统一置 -999。
#    - 日期类型（request_time/fechaConsulta）：
#      * request_time：用 pd.to_datetime 解析；解析失败会变 NaT（后续也会走 A/C/D 的 -999 兜底逻辑）。
#      * fechaConsulta：用 pd.to_datetime 解析；解析失败（NaT）导致 days_before_request 为空，这条查询记录会被过滤掉，不参与统计。
#      * 未来查询（fechaConsulta > request_time）：days_before_request 为负，也会被过滤掉，不参与统计。
#    - 字符串/类别类型（nombreOtorgante/tipoCredito）：
#      * nombreOtorgante：按字典映射到 17 类（新增：个征信机构=SIC）；不在字典或缺失 -> 归为“其他”（不进入 17 类统计，但仍可能计入总查询次数）。
#      * tipoCredito：先 strip + upper 标准化；只统计 CC/PP/TC，其它取值不进入三类统计，但仍可能计入总查询次数。
#    - 数值类型（统计输出）：
#      * cnt：没有记录 -> 0
#      * ratio：分母为 0 时 -> 0
#      * days_mean/days_std：没有记录 -> NaN（因为没数据算均值/标准差）
#      * notnull_ratio：没有记录 -> 0
#      * A/C/D 命中时：所有衍生特征统一 -999（覆盖上面的 0/NaN 结果）。
# =============================================================================

import json  # 导入标准库 json：后续用于把 response_body（JSON 字符串）解析成 Python 对象
from pathlib import Path  # 导入 Path：用面向对象方式处理路径，避免手写字符串路径出错

import numpy as np  # 导入 numpy：后续用于 NaN、数值 replace 等操作（例如把 0 替换为 NaN 再做除法）
import pandas as pd  # 导入 pandas：读取 CSV/PKL、时间解析、DataFrame 操作、groupby 聚合等都依赖它

# 大白话：为了让所有板块口径一致，这里优先读取你筛选后的 pickle：cdc_pickle_pass_fpd7.pkl。
# 大白话：如果你想回到 331 条 CSV，只需要把 USE_PICKLE 改成 False。
USE_PICKLE = True  # True=优先读 pickle；False=读 CSV
INPUT_PICKLE = Path("cdc_pickle_pass_fpd7.pkl")  # 你筛选后的数据

INPUT_CSV = Path("衍生原始数据_331条.csv")  # 兜底：如果 pickle 不存在才读 CSV
OUTPUT_DIR = Path("outputs")  # 输出目录路径（所有导出文件都放在这里）
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # 创建输出目录：parents=True 递归创建；exist_ok=True 已存在不报错

# 大白话：读入数据（优先 pickle；否则读 CSV）。
if USE_PICKLE and INPUT_PICKLE.exists():
    df_raw = pd.read_pickle(INPUT_PICKLE)
    print("[INFO] loaded pickle:", INPUT_PICKLE)
else:
    df_raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    print("[INFO] loaded csv:", INPUT_CSV)

# 大白话：cdc_pickle_pass_fpd7.pkl 里一般用 apply_time 表示“截止日期”，这里统一映射成 request_time。
if ("request_time" not in df_raw.columns) and ("apply_time" in df_raw.columns):
    df_raw["request_time"] = df_raw["apply_time"]
    print("[INFO] mapped apply_time -> request_time")

# 大白话：把截止日期统一解析成 pandas datetime（解析失败 -> NaT）。
df_raw["request_time"] = pd.to_datetime(df_raw.get("request_time"), errors="coerce")

required_cols = ["apply_id", "request_time", "response_body"]  # 定义本流程最小必需字段：申请ID、截止时间、JSON正文
missing = [c for c in required_cols if c not in df_raw.columns]  # 找出缺失列：遍历 required_cols，筛出不在 df_raw.columns 的列名

# === 情况 A：关键字段缺失 ===
# 你的要求：情况 A 直接把“衍生特征”统一返回 -999，而不是报错退出。
# 注意：我们仍然会尽量保留 apply_id 行（如果 apply_id 也缺失，那就只能输出空表）。
FATAL_MISSING_COLUMNS = len(missing) > 0  # 只要 required_cols 中有任何缺失，就认为命中情况 A

if FATAL_MISSING_COLUMNS:  # 如果命中情况 A
    print("[WARN] 命中情况A：关键字段缺失 -> 后续该申请的衍生特征将统一填 -999")  # 打印提示
    print("[WARN] 缺失字段:", missing)  # 打印具体缺失列

    # 为了让后续代码还能运行（并最终输出 -999 特征），我们把缺失列补出来：
    if "apply_id" not in df_raw.columns:  # 如果连 apply_id 都没有
        df_raw["apply_id"] = pd.Series(dtype="int64")  # 创建一个空的 apply_id 列（最终只能输出 0 行）
    if "request_time" not in df_raw.columns:  # 如果缺 request_time
        df_raw["request_time"] = pd.NaT  # 用 NaT（空时间）补齐
    if "response_body" not in df_raw.columns:  # 如果缺 response_body
        df_raw["response_body"] = None  # 用 None 补齐（后续会命中情况 C）

# 解析 request_time（即使部分解析失败也不中断；解析失败的行，后面也会把特征置为 -999）
df_raw["request_time"] = pd.to_datetime(df_raw["request_time"], errors="coerce")  # 将 request_time 转为 datetime；失败则变 NaT（缺失时间）

print("df_raw shape:", df_raw.shape)  # 输出原始表的规模（行数、列数），用于 sanity check

df_raw.head(3)  # 展示前 3 行，快速确认字段值长什么样（尤其 request_time/response_body）


[INFO] loaded pickle: cdc_pickle_pass_fpd7.pkl
[INFO] mapped apply_time -> request_time
df_raw shape: (12546, 12)


,apply_id,response_body,apply_time,approve_state,credit_limit_amount,use_amount,principal_amount_borrowed,fpd7,spd7,credit_apply_cnt,blind_lend,request_time
0,1065991091661283329,"{""claveOtorgante"":""004676"",""consultas"":[{""clav...",2025-11-25 01:53:44.943,CYCLE_PASS,800.00000000,700.00000000,700.00000000,0,0,1,NaN,2025-11-25 01:53:44.943
1,1066560157648134145,"{""claveOtorgante"":""004676"",""consultas"":[{""clav...",2025-11-25 06:29:46.980,CYCLE_PASS,500.00000000,500.00000000,500.00000000,0,0,1,1.0,2025-11-25 06:29:46.980
2,1066719243236777985,"{""claveOtorgante"":""004676"",""consultas"":[{""clav...",2025-11-25 07:46:56.302,SINGLE_PASS,500.00000000,500.00000000,500.00000000,0,0,1,NaN,2025-11-25 07:46:56.302


In [78]:
# 代码格 2/4：处理数据（逐行注释版）  # 单元格标题：说明这是第 2 个代码格，职责是“把 consultas 明细整理出来”
# 目标：
# 1) 从 df_raw.response_body 解析出 consultas 列表  # 处理目标 1：从 JSON 中拿到查询明细
# 2) 把每条 consulta 平铺成一行，形成 consultas_df（apply_id 会重复）  # 处理目标 2：明细表
# 3) 根据你给定的字典，把 nombreOtorgante 归类为 17 个大类（新增：个征信机构=SIC）  # 处理目标 3：机构归类
# 4) 计算 days_before_request = request_time - fechaConsulta（天）  # 处理目标 4：窗口统计所需数值
# 5) 额外：把 tipoCredito 规范化，并对 CC/PP/TC 三类做同样的统计  # 新增目标：对 tipoCredito 三类衍生同样的特征

OTORGANTE_GROUP_DICT = {  # 机构归类字典：key=大类名（中文），value=该大类对应的原始 nombreOtorgante 取值列表
    "商店信息": [  # 大类：商店信息
        "TIENDA COMERCIAL",  # 原始机构字符串：属于“商店信息”的一种
        "TIENDA DE AUTOSERVICIO",  # 原始机构字符串：属于“商店信息”的一种
        "TIENDA DEPARTAMENTAL",  # 原始机构字符串：属于“商店信息”的一种
    ],  # 结束：商店信息
    "大众金融协会": [  # 大类：大众金融协会
        "SOCIEDADES FINANCIERAS POPULARES",  # 原始机构字符串：属于“大众金融协会”的一种
        "SOCIEDAD FINANCIERA COMUNITARIA",  # 原始机构字符串：属于“大众金融协会”的一种
        "ARRENDADORAS FINANCIERAS",  # 原始机构字符串：属于“大众金融协会”的一种
        "UNION DE CREDITO",  # 原始机构字符串：属于“大众金融协会”的一种
    ],  # 结束：大众金融协会
    "多用途": [  # 大类：多用途
        "SOCIEDAD FINANCIERA DE OBJETO MULTIPLE",  # 原始机构字符串：属于“多用途”的一种
    ],  # 结束：多用途
    "非银行抵押": [  # 大类：非银行抵押
        "HIPOTECARIO NO BANCARIO",  # 原始机构字符串：属于“非银行抵押”的一种
    ],  # 结束：非银行抵押
    "服务信息": [  # 大类：服务信息
        "SALUD Y SERVICIOS MEDICOS",  # 原始机构字符串：属于“服务信息”的一种
        "SERVICIO MEDICO",  # 原始机构字符串：属于“服务信息”的一种
        "SERVS. GRALES.",  # 原始机构字符串：属于“服务信息”的一种
        "SERVICIOS",  # 原始机构字符串：属于“服务信息”的一种
    ],  # 结束：服务信息
    "付费电视": [  # 大类：付费电视
        "SERVICIO DE TELEVISION DE PAGA",  # 原始机构字符串：属于“付费电视”的一种
    ],  # 结束：付费电视
    "个人贷款": [  # 大类：个人贷款
        "COMPANIA DE PRESTAMO PERSONAL",  # 原始机构字符串：属于“个人贷款”的一种
        "SOFOL PRESTAMO PERSONAL",  # 原始机构字符串：属于“个人贷款”的一种
    ],  # 结束：个人贷款
    "基金和信托": [  # 大类：基金和信托
        "FONDOS Y FIDEIC",  # 原始机构字符串：属于“基金和信托”的一种
        "FONDOS Y FIDEICOMISOS",  # 原始机构字符串：属于“基金和信托”的一种
        "FONDOS Y FIDEICO",  # 原始机构字符串：属于“基金和信托”的一种
    ],  # 结束：基金和信托
    "建筑": [  # 大类：建筑
        "MERCANCIA PARA LA CONSTRUCCION",  # 原始机构字符串：属于“建筑”的一种
    ],  # 结束：建筑
    "金融公司": [  # 大类：金融公司
        "SOFOL EMPRESARIAL",  # 原始机构字符串：属于“金融公司”的一种
        "SOFOL AUTOMOTRIZ",  # 原始机构字符串：属于“金融公司”的一种
        "OTRAS FINANCIERA",  # 原始机构字符串：属于“金融公司”的一种
        "ARRENDADORAS NO FINANCIERAS",  # 原始机构字符串：属于“金融公司”的一种
    ],  # 结束：金融公司
    "通讯": [  # 大类：通讯
        "TELEFONIA LOCAL Y DE LARGA DISTANCIA",  # 原始机构字符串：属于“通讯”的一种
        "TELEFONIA CELULAR",  # 原始机构字符串：属于“通讯”的一种
        "COMUNICACIONES",  # 原始机构字符串：属于“通讯”的一种
    ],  # 结束：通讯
    "销售": [  # 大类：销售
        "VENTA POR CATALOGO",  # 原始机构字符串：属于“销售”的一种
    ],  # 结束：销售
    "小额贷款": [  # 大类：小额贷款
        "MIC CREDITO PERS",  # 原始机构字符串：属于“小额贷款”的一种
        "MICROFINANCIERA",  # 原始机构字符串：属于“小额贷款”的一种
    ],  # 结束：小额贷款
    "银行": [  # 大类：银行
        "BANCOS",  # 原始机构字符串：属于“银行”的一种
        "BANCO",  # 原始机构字符串：属于“银行”的一种
    ],  # 结束：银行
    "非银行": [  # 大类：非银行
        "FINANCIERA",  # 原始机构字符串：属于“非银行”的一种
    ],  # 结束：非银行
    "政府": [  # 大类：政府
        "GUBERNAMENTALES",  # 原始机构字符串：属于“政府”的一种
        "GOBIERNO",  # 原始机构字符串：属于“政府”的一种
        "HIPOTECAGOBIERNO",  # 原始机构字符串：属于“政府”的一种
    ],  # 结束：政府
    "个征信机构": [  # 大类：个征信机构（你新增）
        "SIC",  # 原始机构字符串：SIC
    ],  # 结束：个征信机构
}  # 结束：OTORGANTE_GROUP_DICT

GROUPS_TO_USE = [  # 明确要衍生特征的 17 个大类（顺序也决定最终特征列的生成顺序）
    "商店信息",  # 17 类之一：商店信息
    "大众金融协会",  # 17 类之一：大众金融协会
    "多用途",  # 17 类之一：多用途
    "非银行抵押",  # 17 类之一：非银行抵押
    "服务信息",  # 17 类之一：服务信息
    "付费电视",  # 17 类之一：付费电视
    "个人贷款",  # 17 类之一：个人贷款
    "基金和信托",  # 17 类之一：基金和信托
    "建筑",  # 17 类之一：建筑
    "金融公司",  # 17 类之一：金融公司
    "通讯",  # 17 类之一：通讯
    "销售",  # 17 类之一：销售
    "小额贷款",  # 17 类之一：小额贷款
    "银行",  # 17 类之一：银行
    "非银行",  # 17 类之一：非银行
    "政府",  # 17 类之一：政府
    "个征信机构",  # 17 类之一：个征信机构（SIC）
]  # 结束：GROUPS_TO_USE

GROUP_NAME_TO_ID = {  # 大类中文名 -> 英文 id（用于生成更规范的特征列名，方便建模/工程化）
    "商店信息": "shop",  # 商店信息 -> shop
    "大众金融协会": "mass_fin_assn",  # 大众金融协会 -> mass_fin_assn
    "多用途": "multi_purpose",  # 多用途 -> multi_purpose
    "非银行抵押": "nonbank_mortgage",  # 非银行抵押 -> nonbank_mortgage
    "服务信息": "service",  # 服务信息 -> service
    "付费电视": "paytv",  # 付费电视 -> paytv
    "个人贷款": "personal_loan",  # 个人贷款 -> personal_loan
    "基金和信托": "fund_trust",  # 基金和信托 -> fund_trust
    "建筑": "construction",  # 建筑 -> construction
    "金融公司": "finance_company",  # 金融公司 -> finance_company
    "通讯": "telecom",  # 通讯 -> telecom
    "销售": "catalog_sale",  # 销售 -> catalog_sale
    "小额贷款": "micro_loan",  # 小额贷款 -> micro_loan
    "银行": "bank",  # 银行 -> bank
    "非银行": "nonbank",  # 非银行 -> nonbank
    "政府": "government",  # 政府 -> government
    "个征信机构": "sic",  # 个征信机构 -> sic
}  # 结束：GROUP_NAME_TO_ID

# tipoCredito 三类口径（你指定的映射）
# CC-消费信贷；PP-个人贷款；TC-信用卡
TIPO_CREDITO_TO_USE = ["CC", "PP", "TC"]  # 需要做特征统计的 tipoCredito 取值
TIPO_CREDITO_TO_ID = {  # tipoCredito -> 列名 id（这里直接用小写代码，避免和机构 personal_loan 重名）
    "CC": "cc",  # 消费信贷
    "PP": "pp",  # 个人贷款
    "TC": "tc",  # 信用卡
}

VALUE_TO_GROUP = {  # 构造“反向映射”：原始 nombreOtorgante 取值 -> 17 大类中文名（查找是 O(1)）
    raw_value: group  # key=原始机构字符串，value=归类后的大类名
    for group, values in OTORGANTE_GROUP_DICT.items()  # 外层：遍历每个大类及其原始值列表
    for raw_value in values  # 内层：遍历该大类下的每个原始值
}  # 结束：VALUE_TO_GROUP


def map_otorgante_group(nombre_otorgante: str) -> str:  # 定义映射函数：把原始 nombreOtorgante 映射到 17 大类；否则归为“其他”
    if pd.isna(nombre_otorgante):  # 如果原始值是 NaN/None（缺失）
        return "其他"  # 缺失则直接归为“其他”
    return VALUE_TO_GROUP.get(str(nombre_otorgante), "其他")  # 用反向映射字典查找；找不到也归为“其他”


rows = []  # 初始化列表：用于收集平铺后的每条 consulta（每条是一个 dict）

# apply_status：记录每个 apply_id 是否命中 A/C/D（用于最后把该申请的衍生特征统一置为 -999）
apply_status = {aid: "ok" for aid in df_raw["apply_id"].tolist()}  # 默认都标记为 ok

# 如果上一格命中了情况 A（关键字段缺失），那这一批 apply_id 都要标记为 A
if "FATAL_MISSING_COLUMNS" in globals() and FATAL_MISSING_COLUMNS:  # 判断全局是否存在该标记且为 True
    for aid in apply_status:  # 遍历所有申请
        apply_status[aid] = "A_missing_columns"  # 标记为情况 A

for apply_id, request_time, response_body in df_raw[["apply_id", "request_time", "response_body"]].itertuples(index=False, name=None):  # 遍历每个申请：取出 apply_id、截止时间、JSON正文
    # 情况 C：response_body 不是合法 JSON（或缺失）
    if response_body is None or (isinstance(response_body, float) and pd.isna(response_body)) or str(response_body).strip() == "":  # response_body 缺失/空字符串
        apply_status[apply_id] = "C_bad_json"  # 标记为情况 C
        continue  # 跳过该申请（该申请最终特征会被统一置 -999）

    try:
        obj = json.loads(response_body) if isinstance(response_body, str) else response_body  # JSON 反序列化：把字符串转成 dict；若已是对象则直接用
    except Exception:
        apply_status[apply_id] = "C_bad_json"  # JSON 解析失败 -> 情况 C
        continue  # 跳过该申请

    # 情况 D：response_body 里没有 consultas 这个块（key 不存在）
    if not isinstance(obj, dict) or ("consultas" not in obj):  # 不是 dict 或者 dict 里没有 consultas key
        apply_status[apply_id] = "D_no_consultas_key"  # 标记为情况 D
        continue  # 跳过该申请

    consultas = obj.get("consultas")  # 取 consultas（此处 key 一定存在，但 value 可能是 None/非 list）

    # 情况 D（扩展）：consultas 存在但不是 list，也视为 consultas 块异常
    if not isinstance(consultas, list):  # 如果 consultas 不是 list
        apply_status[apply_id] = "D_no_consultas_key"  # 仍按情况 D 处理
        continue  # 跳过该申请

    # 正常情况：consultas 是 list（可以为空 list，这种不属于情况 D，只是“没有查询记录”）
    for item in consultas:  # 遍历该申请下的每条 consulta 明细
        rows.append(  # 将需要的字段整理成一行并追加到 rows
            {
                "apply_id": apply_id,  # 申请 id（后续聚合 key）
                "request_time": request_time,  # 截止时间（用于计算 days_before_request）
                "fechaConsulta": item.get("fechaConsulta"),  # 查询日期（原始字段，字符串日期）
                "nombreOtorgante": item.get("nombreOtorgante"),  # 原始机构类型（用于归类）
                "tipoCredito": item.get("tipoCredito"),  # 信贷类别（用于 CC/PP/TC 三类统计）
            }
        )  # 结束：append 一行

consultas_df = pd.DataFrame(rows)  # 将 rows 转为 DataFrame：每条 consulta 一行（apply_id 会重复）

consultas_df["fechaConsulta"] = pd.to_datetime(consultas_df["fechaConsulta"], errors="coerce")  # 解析 fechaConsulta 为 datetime；失败则 NaT

consultas_df["days_before_request"] = (  # 计算 days_before_request（单位：天），作为窗口过滤与均值/标准差的统计对象
    (consultas_df["request_time"] - consultas_df["fechaConsulta"]).dt.total_seconds() / 86400  # 先得到 Timedelta，再转秒并除以 86400 换算为天
)  # 结束：days_before_request

consultas_df = consultas_df[  # 过滤掉不合理/无法统计的记录
    consultas_df["days_before_request"].notna()  # 1) fechaConsulta 解析失败会导致 days_before_request 为 NaN，此处过滤掉
    & (consultas_df["days_before_request"] >= 0)  # 2) fechaConsulta 晚于 request_time 会导致负数（未来查询），此处过滤掉
].copy()  # copy()：避免 SettingWithCopyWarning，并保证后续赋值安全

consultas_df["otorgante_group"] = consultas_df["nombreOtorgante"].apply(map_otorgante_group)  # 对每条明细做机构归类，生成大类字段

# 规范化 tipoCredito：统一转大写、去空格；缺失则置空字符串（便于 isin 判断）
consultas_df["tipo_credito"] = (  # 新增一列：标准化后的 tipoCredito
    consultas_df["tipoCredito"]  # 取原始 tipoCredito 列
    .fillna("")  # 把缺失值填成空字符串，避免后续 str 操作报错
    .astype(str)  # 强制转字符串
    .str.strip()  # 去掉首尾空格
    .str.upper()  # 转大写，保证 CC/PP/TC 的匹配稳定
)

print("consultas_df shape:", consultas_df.shape)  # 打印平铺后的明细表规模（行数、列数）
print("days_before_request 最小值:", float(consultas_df["days_before_request"].min()))  # 打印最小天数差，理论上应 >= 0

print("\n机构归类分布（Top 20）:")  # 打印提示：下面输出归类分布
print(consultas_df["otorgante_group"].value_counts().head(20).to_string())  # 查看大类分布，快速确认映射是否生效

print("\ntipoCredito 分布（Top 20）:")  # 打印提示：下面输出 tipoCredito 的分布
print(consultas_df["tipo_credito"].value_counts().head(20).to_string())  # 查看 CC/PP/TC 是否存在，以及是否有其他取值

consultas_df.head(5)  # 展示前 5 行明细，便于肉眼核对字段


consultas_df shape: (412045, 8)
days_before_request 最小值: 0.00011248842592592591

机构归类分布（Top 20）:
otorgante_group
银行        79345
个征信机构     76511
个人贷款      70521
小额贷款      51509
非银行       41801
多用途       37267
其他        23572
大众金融协会    11950
商店信息       4835
通讯         4529
政府         3541
基金和信托      3369
服务信息       2817
销售          268
金融公司        191
建筑           17
非银行抵押         2

tipoCredito 分布（Top 20）:
tipo_credito
M     152675
F     119872
TC     63447
NC     43069
PP     19966
Q       5499
LC      1966
OT      1311
PG      1185
CA       808
PN       735
AM       643
R        333
E        164
PQ        80
PM        58
AE        57
BR        47
AA        23
P         22


,apply_id,request_time,fechaConsulta,nombreOtorgante,tipoCredito,days_before_request,otorgante_group,tipo_credito
0,1065991091661283329,2025-11-25 01:53:44.943,2025-11-25,CAMELINAS MECANICAS,M,0.078992,其他,M
1,1065991091661283329,2025-11-25 01:53:44.943,2025-11-04,SERVICIOS,M,21.078992,服务信息,M
2,1065991091661283329,2025-11-25 01:53:44.943,2025-10-21,COMPANIA DE PRESTAMO PERSONAL,M,35.078992,个人贷款,M
3,1065991091661283329,2025-11-25 01:53:44.943,2025-09-03,COMPANIA DE PRESTAMO PERSONAL,M,83.078992,个人贷款,M
4,1065991091661283329,2025-11-25 01:53:44.943,2025-09-01,SOCIEDAD FINANCIERA DE OBJETO MULTIPLE,M,85.078992,多用途,M


In [79]:
# 代码格 3/4：衍生函数逻辑（逐行注释版）  # 单元格标题：这里把“明细”聚合成“特征”
# 目标：定义一个可复用函数 derive_consultas_features，用于按多窗口（30/60/.../720天）+ 17类机构归类生成特征表（含新增：个征信机构=SIC）  # 单元格目的


def derive_consultas_features(  # 定义主函数：输入原始表与 consultas 明细，输出按 apply_id 聚合后的特征表
    df_raw: pd.DataFrame,  # 原始表：至少包含 apply_id 与 request_time（用于确定每个申请的截止时间）
    consultas_df: pd.DataFrame,  # consultas 明细表：应包含 apply_id、days_before_request、otorgante_group、tipo_credito 等字段
    window_days_list: list[int],  # 时间窗口列表（单位：天），例如 [30,60,90,...]
    groups_to_use: list[str],  # 机构大类列表（17 类，含新增：个征信机构=SIC），用于决定生成哪些“机构类别”特征
    group_name_to_id: dict[str, str],  # 机构大类中文名到英文 id 的映射（用于生成稳定、可读的列名）
    tipo_credito_to_use: list[str],  # tipoCredito 要统计的取值列表（你要求：CC/PP/TC）
    tipo_credito_to_id: dict[str, str],  # tipoCredito 取值到英文 id 的映射（如 CC->cc）
    apply_status: dict | None = None,  # 每个 apply_id 的状态（命中 A/C/D 时用于把特征统一置为 -999）
    sentinel_value: float = -999,  # 你指定的缺省返回值：-999
) -> pd.DataFrame:  # 返回特征表（index=apply_id，列为各类窗口特征）
    """按 request_time 截止日，对 consultas 按多窗口衍生特征。

    为什么要先把 fechaConsulta 转成数值 days_before_request？
    - 日期本身做均值/方差不直观
    - days_before_request（距截止日多少天）能直接表达“查询新鲜度”，且统计意义清晰

    产出字段（每个 window N）：
    - consultas_{N}d_total_cnt: N天内总查询数量（窗口内所有 consultas，不区分是否落入17类）
    - consultas_{N}d_{group}_cnt: N天内该类别数量
    - consultas_{N}d_{group}_ratio: 该类别数量 / 总查询数量
    - consultas_{N}d_{group}_days_mean: 该类别 days_before_request 均值（越小代表越近期）
    - consultas_{N}d_{group}_days_std: 该类别 days_before_request 标准差（离散程度）
    - consultas_{N}d_{group}_notnull_ratio: days_before_request 非空占比（一般用于质量监控）
    """

    df_base = (  # 构造基表：每个 apply_id 一行，并保留 request_time（截止时间）
        df_raw[["apply_id", "request_time"]]  # 只取出后续需要的两列
        .drop_duplicates("apply_id")  # 若同一 apply_id 在原始表出现多行，这里去重保证一行一个 apply_id
        .set_index("apply_id")  # 以 apply_id 作为索引，方便后续按索引 join 各窗口特征
        .sort_index()  # 对索引排序，方便查看（对计算结果没有影响）
    )  # 结束：df_base

    features = df_base.copy()  # 初始化特征表：先复制基表，后续不断 join 新特征列

    def id_of(group_name: str) -> str:  # 小工具函数：把中文大类名转换为英文 id（用于列名）
        return group_name_to_id.get(group_name, group_name)  # 若字典中不存在映射，则退回使用原字符串

    def build_features_for_window(window_days: int) -> pd.DataFrame:  # 针对单个窗口 N 天构造特征（返回 index=apply_id 的 DataFrame）
        sub = consultas_df[  # 在 consultas 明细中筛选“落在窗口内”的记录
            (consultas_df["days_before_request"] >= 0)  # 条件1：距截止日天数差非负（未来查询已在前面过滤，一般这里是冗余保护）
            & (consultas_df["days_before_request"] <= window_days)  # 条件2：距截止日不超过 window_days（即 N 天内）
        ].copy()  # copy：避免链式赋值问题

        out = pd.DataFrame(index=df_base.index)  # 初始化输出：先建一个只有索引（apply_id）的空表，用来装该窗口的所有特征列

        total = (  # 计算总查询数量：窗口内所有 consultas 的条数（包含不在17类内的“其他”）
            sub.groupby("apply_id")  # 按 apply_id 分组
            .size()  # 每组的行数就是查询次数
            .reindex(df_base.index)  # 对齐到所有 apply_id（没有查询的也要有一行）
            # zlf update: 特征值为空时填充-999
            .fillna(-999)  # 缺失填 0，表示该 apply_id 在该窗口内没有 consultas
            .astype(int)  # 转成 int，便于下游使用
        )  # 结束：total

        out[f"consultas_{window_days}d_total_cnt"] = total  # 写入“总查询数量”特征列

        # === 机构 17 大类特征（含新增：个征信机构=SIC） ===
        sub_cat = sub[sub["otorgante_group"].isin(groups_to_use)].copy()  # 仅保留落入 17 个大类的记录（用于生成“机构类别”特征）

        if len(sub_cat) == 0:  # 如果窗口内没有任何命中 17 类的记录，则所有“机构类别特征”都需要补默认值
            for g in groups_to_use:  # 遍历 17 个大类，逐一补齐列
                gid = id_of(g)  # 取该大类对应的英文 id
                out[f"consultas_{window_days}d_{gid}_cnt"] = 0  # 该类数量：0
                out[f"consultas_{window_days}d_{gid}_ratio"] = 0.0  # 该类占比：0
                # zlf update: 特征值为空时填充-999
                out[f"consultas_{window_days}d_{gid}_days_mean"] = -999  # 均值：无数据用 -999
                # zlf update: 特征值为空时填充-999
                out[f"consultas_{window_days}d_{gid}_days_std"] = -999  # 标准差：无数据用 -999
                out[f"consultas_{window_days}d_{gid}_notnull_ratio"] = 0.0  # notNull 占比：0（没有记录）
        else:
            g = sub_cat.groupby(["apply_id", "otorgante_group"])  # 对“apply_id × 机构大类”做分组，后续可一次性算出每组的统计量

            cnt = (  # 该类别数量：每个 apply_id 在该大类下的记录条数
                g.size()  # 分组计数
                .unstack()  # 把第二层索引（大类）展开成列
                .reindex(index=df_base.index, columns=groups_to_use)  # 对齐行列，保证每个 apply_id 和每个大类都有位置
            )  # 结束：cnt
            # zlf update: 特征值为空时填充-999
            cnt = cnt.fillna(0).astype(int)  # 没有记录的位置填 0，并转 int

            mean = (
                g["days_before_request"].mean().unstack().reindex(index=df_base.index, columns=groups_to_use)
            )  # days_before_request 的均值
            # zlf update: 特征值为空时填充-999
            mean = mean.fillna(-999.0)

            std = (
                g["days_before_request"].std(ddof=0).unstack().reindex(index=df_base.index, columns=groups_to_use)
            )  # days_before_request 的标准差
            # zlf update: 特征值为空时填充-999
            std = std.fillna(-999.0)

            valid = (
                g["days_before_request"].count().unstack().reindex(index=df_base.index, columns=groups_to_use)
            )  # 非空数量
            # zlf update: 特征值为空时填充-999
            valid = valid.fillna(-999)  # 无记录填 0

            # zlf update: 特征值为空时填充-999
            notnull_ratio = valid.div(cnt.replace(0, np.nan)).fillna(-999.0)  # notNull 占比
            # zlf update: 特征值为空时填充-999
            ratio = cnt.div(total.replace(0, np.nan), axis=0).fillna(-999.0)  # 该类/总查询数量

            # === 新增：每天平均次数 & 每天次数标准差（你的例子：3次/30天=0.1）===
            # 大白话：这组特征不看“离截止日多少天的均值/方差”，而是看“近N天内每天查了几次”的强度与波动。
            # 做法：把 days_before_request 向下取整成 day_bin（0..N-1），统计每个 day_bin 的次数；
            # - daily_cnt_mean = cnt/N（没发生的天按 0 计入均值）
            # - daily_cnt_std  = std(x_1..x_N)（同样把没发生的天当 0；总体标准差 ddof=0）
            sub_cat_day = sub_cat.copy()
            sub_cat_day["day_bin"] = np.floor(sub_cat_day["days_before_request"]).astype("int64")
            sub_cat_day = sub_cat_day[(sub_cat_day["day_bin"] >= 0) & (sub_cat_day["day_bin"] < window_days)]

            if len(sub_cat_day) == 0:
                daily_mean = pd.DataFrame(0.0, index=df_base.index, columns=groups_to_use)
                daily_std = pd.DataFrame(0.0, index=df_base.index, columns=groups_to_use)
            else:
                day_cnt = (
                    sub_cat_day.groupby(["apply_id", "otorgante_group", "day_bin"]).size().rename("day_cnt")
                )
                # sumsq: (x1^2 + x2^2 + ...)，只对“有记录的天”求和；没记录的天默认 0
                sumsq = (day_cnt**2).groupby(["apply_id", "otorgante_group"]).sum().unstack()
                # zlf update: 特征值为空时填充-999
                sumsq = sumsq.reindex(index=df_base.index, columns=groups_to_use).fillna(0.0)

                # cnt_in_Ndays: 近N天（按 day_bin< N）内该类别总次数
                cnt_in_Ndays = day_cnt.groupby(["apply_id", "otorgante_group"]).sum().unstack()
                # zlf update: 特征值为空时填充0（用于计算）
                cnt_in_Ndays = cnt_in_Ndays.reindex(index=df_base.index, columns=groups_to_use).fillna(0.0)

                daily_mean = cnt_in_Ndays / float(window_days)
                var = (sumsq / float(window_days)) - (daily_mean**2)
                daily_std = np.sqrt(var.clip(lower=0.0))

            for group_name in groups_to_use:  # 遍历每个大类，把对应列写到 out（列名用英文 id）
                gid = id_of(group_name)  # 大类英文 id
                out[f"consultas_{window_days}d_{gid}_cnt"] = cnt[group_name]  # 写入：该类数量
                out[f"consultas_{window_days}d_{gid}_ratio"] = ratio[group_name]  # 写入：该类占比
                out[f"consultas_{window_days}d_{gid}_days_mean"] = mean[group_name]  # 写入：days 均值
                out[f"consultas_{window_days}d_{gid}_days_std"] = std[group_name]  # 写入：days 标准差
                out[f"consultas_{window_days}d_{gid}_notnull_ratio"] = notnull_ratio[group_name]  # 写入：notNull 占比
                out[f"consultas_{window_days}d_{gid}_daily_cnt_mean"] = daily_mean[group_name]  # 新增：每天平均次数（cnt/N）
                out[f"consultas_{window_days}d_{gid}_daily_cnt_std"] = daily_std[group_name]  # 新增：每天次数标准差

        # === tipoCredito 三类特征（CC/PP/TC）===
        # 说明：tipo_credito 是在“处理数据”代码格里做过 strip+upper 的标准化字段
        sub_tipo = sub[sub["tipo_credito"].isin(tipo_credito_to_use)].copy()  # 仅保留 CC/PP/TC

        if len(sub_tipo) == 0:  # 如果窗口内没有任何 CC/PP/TC 记录，则补默认值
            for t in tipo_credito_to_use:  # 遍历三类
                tid = tipo_credito_to_id.get(t, t.lower())  # 取英文 id（比如 CC->cc）
                out[f"consultas_{window_days}d_tipo_{tid}_cnt"] = 0
                out[f"consultas_{window_days}d_tipo_{tid}_ratio"] = 0.0
                # zlf update: 特征值为空时填充-999
                out[f"consultas_{window_days}d_tipo_{tid}_days_mean"] = -999
                # zlf update: 特征值为空时填充-999
                out[f"consultas_{window_days}d_tipo_{tid}_days_std"] = -999
                out[f"consultas_{window_days}d_tipo_{tid}_notnull_ratio"] = 0.0
        else:
            gt = sub_tipo.groupby(["apply_id", "tipo_credito"])  # 对“apply_id × tipo_credito”分组

            # zlf update: 特征值为空时填充-999
            cnt_t = gt.size().unstack().reindex(index=df_base.index, columns=tipo_credito_to_use).fillna(0).astype(int)
            mean_t = gt["days_before_request"].mean().unstack().reindex(index=df_base.index, columns=tipo_credito_to_use)
            mean_t = mean_t.fillna(-999.0)

            std_t = gt["days_before_request"].std(ddof=0).unstack().reindex(index=df_base.index, columns=tipo_credito_to_use)
            std_t = std_t.fillna(-999.0)

            # zlf update: 特征值为空时填充-999
            valid_t = gt["days_before_request"].count().unstack().reindex(index=df_base.index, columns=tipo_credito_to_use).fillna(-999)

            # zlf update: 特征值为空时填充-999
            notnull_ratio_t = valid_t.div(cnt_t.replace(0, np.nan)).fillna(-999.0)
            # zlf update: 特征值为空时填充-999
            ratio_t = cnt_t.div(total.replace(0, np.nan), axis=0).fillna(-999.0)

            # === 新增：tipoCredito 每天平均次数 & 每天次数标准差 ===
            sub_tipo_day = sub_tipo.copy()
            sub_tipo_day["day_bin"] = np.floor(sub_tipo_day["days_before_request"]).astype("int64")
            sub_tipo_day = sub_tipo_day[(sub_tipo_day["day_bin"] >= 0) & (sub_tipo_day["day_bin"] < window_days)]

            if len(sub_tipo_day) == 0:
                daily_mean_t = pd.DataFrame(0.0, index=df_base.index, columns=tipo_credito_to_use)
                daily_std_t = pd.DataFrame(0.0, index=df_base.index, columns=tipo_credito_to_use)
            else:
                day_cnt_t = (
                    sub_tipo_day.groupby(["apply_id", "tipo_credito", "day_bin"]).size().rename("day_cnt")
                )
                sumsq_t = (day_cnt_t**2).groupby(["apply_id", "tipo_credito"]).sum().unstack()
                # zlf update: 特征值为空时填充-999
                sumsq_t = sumsq_t.reindex(index=df_base.index, columns=tipo_credito_to_use).fillna(0.0)

                cnt_in_Ndays_t = day_cnt_t.groupby(["apply_id", "tipo_credito"]).sum().unstack()
                # zlf update: 特征值为空时填充-999
                cnt_in_Ndays_t = cnt_in_Ndays_t.reindex(index=df_base.index, columns=tipo_credito_to_use).fillna(0.0)

                daily_mean_t = cnt_in_Ndays_t / float(window_days)
                var_t = (sumsq_t / float(window_days)) - (daily_mean_t**2)
                daily_std_t = np.sqrt(var_t.clip(lower=0.0))

            for t in tipo_credito_to_use:
                tid = tipo_credito_to_id.get(t, t.lower())
                out[f"consultas_{window_days}d_tipo_{tid}_cnt"] = cnt_t[t]
                out[f"consultas_{window_days}d_tipo_{tid}_ratio"] = ratio_t[t]
                out[f"consultas_{window_days}d_tipo_{tid}_days_mean"] = mean_t[t]
                out[f"consultas_{window_days}d_tipo_{tid}_days_std"] = std_t[t]
                out[f"consultas_{window_days}d_tipo_{tid}_notnull_ratio"] = notnull_ratio_t[t]
                out[f"consultas_{window_days}d_tipo_{tid}_daily_cnt_mean"] = daily_mean_t[t]
                out[f"consultas_{window_days}d_tipo_{tid}_daily_cnt_std"] = daily_std_t[t]

        return out  # 返回该窗口结果（index=apply_id）

    for w in window_days_list:  # 遍历所有窗口天数
        w_feat = build_features_for_window(w)  # 计算该窗口的所有特征
        features = features.join(  # 把该窗口特征按 apply_id（索引）拼到总特征表上
            w_feat.drop(columns=["request_time"], errors="ignore"),  # 防御：如果 w_feat 里意外带了 request_time 列则去掉
            how="left",  # 左连接：以 features 的 apply_id 为准，保证不丢任何申请
        )  # 结束：join

    # === 按你的要求处理 A/C/D：衍生特征统一返回 -999 ===
    # 说明：我们只覆盖“衍生出来的特征列”，不覆盖 request_time（它是原始截止时间，不属于衍生特征）。
    if apply_status is not None:  # 如果传入了状态字典
        bad_ids = [  # 收集命中 A/C/D 的 apply_id
            aid
            for aid, st in apply_status.items()
            if st in {"A_missing_columns", "C_bad_json", "D_no_consultas_key"}
        ]
        if len(bad_ids) > 0:  # 如果确实有坏样本
            feature_cols = [c for c in features.columns if c != "request_time"]  # 除 request_time 外的列都视为“衍生特征”
            features.loc[bad_ids, feature_cols] = sentinel_value  # 把这些 apply_id 的所有衍生特征列统一置为 -999

    return features  # 返回最终特征表（含 request_time + 多窗口特征列）


In [80]:
features


,request_time,cdc_consultas_30d_total_cnt_v2,cdc_consultas_30d_shop_cnt_v2,cdc_consultas_30d_shop_ratio_v2,cdc_consultas_30d_shop_days_mean_v2,cdc_consultas_30d_shop_days_std_v2,cdc_consultas_30d_shop_notnull_ratio_v2,cdc_consultas_30d_shop_daily_cnt_mean_v2,cdc_consultas_30d_shop_daily_cnt_std_v2,cdc_consultas_30d_mass_fin_assn_cnt_v2,...,cdc_consultas_720d_tipo_pp_notnull_ratio_v2,cdc_consultas_720d_tipo_pp_daily_cnt_mean_v2,cdc_consultas_720d_tipo_pp_daily_cnt_std_v2,cdc_consultas_720d_tipo_tc_cnt_v2,cdc_consultas_720d_tipo_tc_ratio_v2,cdc_consultas_720d_tipo_tc_days_mean_v2,cdc_consultas_720d_tipo_tc_days_std_v2,cdc_consultas_720d_tipo_tc_notnull_ratio_v2,cdc_consultas_720d_tipo_tc_daily_cnt_mean_v2,cdc_consultas_720d_tipo_tc_daily_cnt_std_v2
apply_id,,,,,,,,,,,,,,,,,,,,,
1065462364007251969,2025-11-24 21:37:16.301,6,0,0.0,-999.0,-999.0,-999.0,0.0,0.0,0,...,-999.0,0.000000,0.000000,8,0.258065,176.400883,155.972754,1.0,0.011111,0.104822
1065471950374248449,2025-11-24 21:41:55.655,2,0,0.0,-999.0,-999.0,-999.0,0.0,0.0,0,...,1.0,0.004167,0.064415,11,0.289474,382.540480,166.498852,1.0,0.015278,0.143527
1065474664793579521,2025-11-24 21:43:14.452,6,0,0.0,-999.0,-999.0,-999.0,0.0,0.0,0,...,1.0,0.005556,0.117720,10,0.250000,157.805028,171.723295,1.0,0.013889,0.138750
1065474733513048065,2025-11-24 21:43:16.834,11,0,0.0,-999.0,-999.0,-999.0,0.0,0.0,0,...,1.0,0.004167,0.064415,20,0.270270,245.055056,196.137012,1.0,0.027778,0.164336
1065474836592279553,2025-11-24 21:43:19.638,2,0,0.0,-999.0,-999.0,-999.0,0.0,0.0,0,...,-999.0,0.000000,0.000000,7,0.184211,371.762231,188.266095,1.0,0.009722,0.098121
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1130960443472683009,2025-12-16 23:08:01.506,4,0,0.0,-999.0,-999.0,-999.0,0.0,0.0,0,...,1.0,0.001389,0.037242,0,0.000000,-999.000000,-999.000000,-999.0,0.000000,0.000000
1131002877749559297,2025-12-16 23:28:36.347,14,0,0.0,-999.0,-999.0,-999.0,0.0,0.0,0,...,1.0,0.006944,0.098356,3,0.060000,48.644865,20.805982,1.0,0.004167,0.064415
1131003427505373185,2025-12-16 23:28:52.500,6,0,0.0,-999.0,-999.0,-999.0,0.0,0.0,1,...,-999.0,0.000000,0.000000,2,0.142857,145.978385,141.000000,1.0,0.002778,0.052631


In [81]:
# 代码格 4/4：生成结果（逐行注释版）  # 单元格标题：这里真正跑衍生并（按需）把结果落盘
# 目标：调用 derive_consultas_features 生成特征表 features；你确认后再导出 2 份 CSV（明细 + 特征）  # 单元格目的

WINDOW_DAYS = [30, 60, 90, 120, 180, 360, 720]  # 定义时间窗口（单位：天）：分别统计截止日前 N 天内的查询特征

features = derive_consultas_features(  # 调用上一格定义的函数，生成按 apply_id 聚合的特征表
    df_raw=df_raw,  # 传入原始表（含 apply_id 与 request_time）
    consultas_df=consultas_df,  # 传入处理后的 consultas 明细表
    window_days_list=WINDOW_DAYS,  # 传入窗口列表
    groups_to_use=GROUPS_TO_USE,  # 传入需要衍生的 17 大类（含新增：个征信机构=SIC）
    group_name_to_id=GROUP_NAME_TO_ID,  # 传入中文->英文 id 映射（用于列名）
    tipo_credito_to_use=TIPO_CREDITO_TO_USE,  # 传入 tipoCredito 三类（CC/PP/TC）
    tipo_credito_to_id=TIPO_CREDITO_TO_ID,  # 传入 tipoCredito -> 列名 id 映射
    apply_status=apply_status,  # 传入每个 apply_id 的状态（命中 A/C/D 的会把特征统一置 -999）
    sentinel_value=-999,  # 你指定的 sentinel 值
)  # 结束：features 计算

# === 输出开关（默认不输出，等你确认后再改成 True）===
WRITE_OUTPUTS = True  # True：写 outputs/*.csv；False：只在内存里生成 features/consultas_df 不落盘
WRITE_FLAT_CSV = False  # zlf update: 暂不输出明细文件，只输出特征文件
# WRITE_SAMPLE_200 = True  # True：输出前200条记录的小文件；False：跳过样本输出
BATCH_SIZE = 500  # zlf update: 改为分批输出，每个文件500条数据
WRITE_FULL_CSV = True  # zlf update: 是否输出全量CSV文件（包含所有数据）

# 你最终只需要 3 个 outputs：
# 1) 原始数据展开后的明细（consultas_flat.csv）
# 2) 衍生后的特征表（cdc1_features_consultas.csv）
# 3) 前200条记录的小文件（cdc1_features_consultas_sample200.csv）- 用于快速查看和测试

consultas_out_path = OUTPUT_DIR / "consultas_flat.csv"  # 定义“平铺明细表”的输出路径
# features_out_path = OUTPUT_DIR / "cdc1_features_consultas.csv"  # 定义“特征表”的输出路径
# features_sample200_path = OUTPUT_DIR / "cdc1_features_consultas_sample200.csv"  # 定义"前200条样本"的输出路径

if WRITE_OUTPUTS:  # 只有你确认后才真正落盘
    if WRITE_FLAT_CSV:  # 根据 WRITE_FLAT_CSV 开关决定是否输出平铺明细文件
        consultas_df.to_csv(consultas_out_path, index=False, encoding="utf-8-sig")  # 导出明细表

    # 只对“衍生特征列”加前缀/后缀（request_time 不加），并写出到 CSV
    _rename_map_for_write = {c: f"cdc_{c}_v2" for c in features.columns if c != "request_time"}

    _features_to_write = features.rename(columns=_rename_map_for_write).reset_index()  # apply_id 变回普通列，便于写 CSV
    _round_cols = [c for c in _features_to_write.columns if c not in {"apply_id", "request_time"}]  # 只对衍生特征列做保留6位小数
    _features_to_write[_round_cols] = _features_to_write[_round_cols].apply(pd.to_numeric, errors="coerce").round(6).fillna(-999.0)  # zlf update: round后再次填充NaN为-999

    # zlf update: 分批输出功能 - 每个文件500条数据
    total_rows = len(_features_to_write)
    num_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

    print("[INFO] 开始分批输出特征文件")
    print(f"[INFO] 总数据量: {total_rows} 行")
    print(f"[INFO] 批次大小: {BATCH_SIZE} 行/文件")
    print(f"[INFO] 输出文件数: {num_batches} 个")
    print()

    batch_files = []

    for batch_idx in range(num_batches):
        start_idx = batch_idx * BATCH_SIZE
        end_idx = min((batch_idx + 1) * BATCH_SIZE, total_rows)
        
        batch_data = _features_to_write.iloc[start_idx:end_idx]
        
        batch_filename = f"cdc1_features_batch{batch_idx + 1:03d}_{start_idx + 1}-{end_idx}.csv"
        batch_path = OUTPUT_DIR / batch_filename
        
        batch_data.to_csv(batch_path, index=False, encoding="utf-8-sig")
        batch_files.append(batch_path)
        
        print(f"[WRITE] 批次 {batch_idx + 1}/{num_batches}: {batch_filename}")
        print(f"        行范围: {start_idx + 1} - {end_idx} ({len(batch_data)} 行)")

    print()
    print("[SUCCESS] 分批输出完成！")
    print("written:")
    if WRITE_FLAT_CSV:
        print("-", consultas_out_path)
    for batch_file in batch_files:
        print("-", batch_file)
    
    # zlf update: 输出全量CSV文件
    if WRITE_FULL_CSV:
        print()
        print("[INFO] 开始输出全量特征文件")
        full_filename = "cdc1_features_all.csv"
        full_path = OUTPUT_DIR / full_filename
        _features_to_write.to_csv(full_path, index=False, encoding="utf-8-sig")
        print(f"[WRITE] 全量文件: {full_filename}")
        print(f"        总行数: {total_rows} 行")
        print("[SUCCESS] 全量输出完成！")
        print("written:")
        print("-", full_path)
else:
    print("[INFO] 当前 WRITE_OUTPUTS=False：已计算完成，但未写出 outputs/*.csv（等你确认后再输出）")

print("consultas_flat rows:", len(consultas_df))  # 打印明细表行数（应等于所有 apply_id 的 consultas 条数之和）
print("features shape:", features.shape)  # 打印特征表形状（行数=apply_id 数；列数=1个request_time+多窗口特征）

# === 特征数据字典（不落盘，仅用于核对/拼接）===
# 说明：这里输出的是“特征列名 -> 含义”的映射表（feature_dict）。
# 三个板块都会按同一字段格式产出 feature_dict，因此可以直接上下拼接：
# pd.concat([fd1, fd2, fd3], ignore_index=True)

import re


def build_feature_dict_block1(cols: list[str]) -> pd.DataFrame:
    rows = []

    for col in cols:
        if col == "request_time":
            continue

        m = re.match(r"^consultas_(\d+)d_total_cnt$", col)
        if m:
            w = int(m.group(1))
            rows.append(
                {
                    "block": "block1_consultas",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "total",
                    "group_code": "ALL",
                    "metric": "count",
                    "stat": "cnt",
                    "description": f"第一板块：近{w}天内 consultas 总查询次数",
                }
            )
            continue

        m = re.match(r"^consultas_(\d+)d_tipo_([^_]+)_(cnt|ratio|days_mean|days_std|notnull_ratio|daily_cnt_mean|daily_cnt_std)$", col)
        if m:
            w = int(m.group(1))
            tid = m.group(2)
            k = m.group(3)
            desc_map = {
                "cnt": "次数",
                "ratio": "占比（该类/总查询次数）",
                "days_mean": "距截止日天数均值",
                "days_std": "距截止日天数标准差",
                "notnull_ratio": "有效值占比",
                "daily_cnt_mean": "每天平均次数（cnt/窗口天数）",
                "daily_cnt_std": "每天次数标准差（按窗口内每天计数序列）",
            }

            if k in {"daily_cnt_mean", "daily_cnt_std"}:
                metric = "daily_count"
            elif k.startswith("days") or k == "notnull_ratio":
                metric = "days_before_request"
            else:
                metric = "count"

            rows.append(
                {
                    "block": "block1_consultas",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "tipo_credito",
                    "group_code": tid.upper(),
                    "metric": metric,
                    "stat": k,
                    "description": f"第一板块：近{w}天内 tipoCredito={tid.upper()} 的{desc_map.get(k,k)}",
                }
            )
            continue

        m = re.match(r"^consultas_(\d+)d_(.+)_(cnt|ratio|days_mean|days_std|notnull_ratio|daily_cnt_mean|daily_cnt_std)$", col)
        if m:
            w = int(m.group(1))
            gid = m.group(2)
            k = m.group(3)
            if gid.startswith("tipo_"):
                continue
            desc_map = {
                "cnt": "次数",
                "ratio": "占比（该类/总查询次数）",
                "days_mean": "距截止日天数均值",
                "days_std": "距截止日天数标准差",
                "notnull_ratio": "有效值占比",
                "daily_cnt_mean": "每天平均次数（cnt/窗口天数）",
                "daily_cnt_std": "每天次数标准差（按窗口内每天计数序列）",
            }

            if k in {"daily_cnt_mean", "daily_cnt_std"}:
                metric = "daily_count"
            elif k.startswith("days") or k == "notnull_ratio":
                metric = "days_before_request"
            else:
                metric = "count"

            rows.append(
                {
                    "block": "block1_consultas",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "otorgante_group",
                    "group_code": gid,
                    "metric": metric,
                    "stat": k,
                    "description": f"第一板块：近{w}天内 机构大类={gid} 的{desc_map.get(k,k)}",
                }
            )
            continue

        # 没解析到的列也保留，避免漏列
        rows.append(
            {
                "block": "block1_consultas",
                "feature_name": col,
                "window_days": None,
                "group_type": "unknown",
                "group_code": None,
                "metric": None,
                "stat": None,
                "description": "第一板块：未匹配到解析规则（请人工确认列名口径）",
            }
        )

    return pd.DataFrame(
        rows,
        columns=[
            "block",
            "feature_name",
            "window_days",
            "group_type",
            "group_code",
            "metric",
            "stat",
            "description",
        ],
    )


feature_dict = build_feature_dict_block1(list(features.columns))
print("feature_dict shape:", feature_dict.shape)
feature_dict.head(10)

# === 导出：特征字典 CSV（3列：特征名称/中文释义/逻辑解释）===
# 说明：你要的是每个板块各自导出一份“特征字典CSV”，用于你后续手工上下拼接。
# 默认不写文件：你确认后把 WRITE_FEATURE_DICT_OUTPUTS 改成 True 才会写出。
WRITE_FEATURE_DICT_OUTPUTS = True

# 大白话：为了让“特征字典”给不熟悉字段的人也看得懂，这里补一份“原始字段 -> 中文含义”的说明。
# 说明：只要 logic_desc 里出现字段名，我们就用“字段名=中文含义”的方式把它解释清楚。
RAW_FIELD_DESC_BLOCK1 = {
    "response_body.consultas": "查询记录列表",
    "fechaConsulta": "查询日期",
    "nombreOtorgante": "机构类别（原始机构字符串）",
    "tipoCredito": "信贷类别（CC/PP/TC等）",
    "request_time": "截止日期（本次申请时间）",
}

# 大白话：统一补充“统计口径/公式”的解释（避免只有英文缩写）。
STAT_DESC_BLOCK1 = {
    "cnt": "数量（窗口内该组记录数）",
    "ratio": "占比=该组cnt/窗口内总cnt",
    "notnull_ratio": "非空占比=该组中该指标非空条数/该组总条数",
    "days_mean": "均值=mean(days_before_request)，days_before_request=(request_time-fechaConsulta)/86400",
    "days_std": "标准差=std(days_before_request)，days_before_request=(request_time-fechaConsulta)/86400",
    "daily_cnt_mean": "日均次数=该组cnt/window_days",
    "daily_cnt_std": "按日计数标准差=std(每天该组查询次数)",
}

# 大白话：把一行特征结构化信息拼成“别人看得懂”的逻辑说明。
def _logic_desc_block1(r) -> str:
    w = r.get("window_days")
    gtype = r.get("group_type")
    gcode = r.get("group_code")
    metric = r.get("metric")
    stat = r.get("stat")

    # 核心窗口口径
    window_desc = (
        f"窗口={w}天（基于{RAW_FIELD_DESC_BLOCK1['fechaConsulta']}={ 'fechaConsulta' }，先算days_before_request=(request_time-fechaConsulta)/86400，再取0<=days_before_request<=window）"
    )

    # 分组口径
    group_desc = f"分组={gtype}:{gcode}（来源字段：nombreOtorgante=机构类别 / tipoCredito=信贷类别）"

    # 指标/统计解释
    stat_explain = STAT_DESC_BLOCK1.get(str(stat), str(stat))
    metric_desc = f"指标={metric}; 统计={stat}（{stat_explain}）"

    # 原始字段释义补充（出现字段名就解释）
    raw_fields = (
        "原始字段释义："
        "response_body.consultas=查询记录列表；"
        "fechaConsulta=查询日期；"
        "nombreOtorgante=机构类别；"
        "tipoCredito=信贷类别；"
        "request_time=截止日期"
    )

    return f"{window_desc}; {group_desc}; {metric_desc}; {raw_fields}"

feature_dict_3col = pd.DataFrame(
    {
        "feature_name": feature_dict["feature_name"],
        "cn_desc": feature_dict["description"],
        "logic_desc": feature_dict.apply(_logic_desc_block1, axis=1),
    }
)

feature_dict_out_path = OUTPUT_DIR / "block1_feature_dict_consultas.csv"

if WRITE_FEATURE_DICT_OUTPUTS:
    feature_dict_3col.to_csv(feature_dict_out_path, index=False, encoding="utf-8-sig")
    print("written feature_dict:")
    print("-", feature_dict_out_path)
else:
    print("[INFO] 当前 WRITE_FEATURE_DICT_OUTPUTS=False：未写出特征字典CSV（等你确认后再输出）")

# === 特征列名统一加前后缀（只改 features，不改特征字典）===
FEATURE_PREFIX = "cdc_"
FEATURE_SUFFIX = "_v2"

rename_map = {
    c: f"{FEATURE_PREFIX}{c}{FEATURE_SUFFIX}"
    for c in features.columns
    if c != "request_time"
}
features = features.rename(columns=rename_map)

# === 衍生特征统一保留6位小数（只改 features，不改特征字典；request_time 不动）===
_round_cols = [c for c in features.columns if c != "request_time"]
features[_round_cols] = features[_round_cols].apply(pd.to_numeric, errors="coerce").round(6).fillna(-999.0)  # zlf update: round后再次填充NaN为-999

features.head(3)  # 展示前 3 行特征，便于快速确认结果


# 对浮点数列进行处理：保留6位小数，空值填充为-999
exclude_cols = ['apply_id', 'request_time', 'apply_time']  # 排除列名，可添加
float_cols = [c for c in feature_dict.select_dtypes(include=['float64', 'float32']).columns if c not in exclude_cols]
feature_dict[float_cols] = feature_dict[float_cols].round(6).fillna(-999)
print("feature_dict shape:", feature_dict.shape)
display(feature_dict.head(10))


/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_20472/3326467059.py:142: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"consultas_{window_days}d_{gid}_ratio"] = ratio[group_name]  # 写入：该类占比
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_20472/3326467059.py:143: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"consultas_{window_days}d_{gid}_days_mean"] = mean[group_name]  # 写入：days 均值
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_20472/3326467059.py:144: PerformanceWarning: DataFrame is

[INFO] 开始分批输出特征文件
[INFO] 总数据量: 12546 行
[INFO] 批次大小: 500 行/文件
[INFO] 输出文件数: 26 个

[WRITE] 批次 1/26: cdc1_features_batch001_1-500.csv
        行范围: 1 - 500 (500 行)
[WRITE] 批次 2/26: cdc1_features_batch002_501-1000.csv
        行范围: 501 - 1000 (500 行)
[WRITE] 批次 3/26: cdc1_features_batch003_1001-1500.csv
        行范围: 1001 - 1500 (500 行)
[WRITE] 批次 4/26: cdc1_features_batch004_1501-2000.csv
        行范围: 1501 - 2000 (500 行)
[WRITE] 批次 5/26: cdc1_features_batch005_2001-2500.csv
        行范围: 2001 - 2500 (500 行)
[WRITE] 批次 6/26: cdc1_features_batch006_2501-3000.csv
        行范围: 2501 - 3000 (500 行)
[WRITE] 批次 7/26: cdc1_features_batch007_3001-3500.csv
        行范围: 3001 - 3500 (500 行)
[WRITE] 批次 8/26: cdc1_features_batch008_3501-4000.csv
        行范围: 3501 - 4000 (500 行)
[WRITE] 批次 9/26: cdc1_features_batch009_4001-4500.csv
        行范围: 4001 - 4500 (500 行)
[WRITE] 批次 10/26: cdc1_features_batch010_4501-5000.csv
        行范围: 4501 - 5000 (500 行)
[WRITE] 批次 11/26: cdc1_features_batch011_5001-5500.csv
   

,block,feature_name,window_days,group_type,group_code,metric,stat,description
0,block1_consultas,consultas_30d_total_cnt,30,total,ALL,count,cnt,第一板块：近30天内 consultas 总查询次数
1,block1_consultas,consultas_30d_shop_cnt,30,otorgante_group,shop,count,cnt,第一板块：近30天内 机构大类=shop 的次数
2,block1_consultas,consultas_30d_shop_ratio,30,otorgante_group,shop,count,ratio,第一板块：近30天内 机构大类=shop 的占比（该类/总查询次数）
3,block1_consultas,consultas_30d_shop_days_mean,30,otorgante_group,shop,days_before_request,days_mean,第一板块：近30天内 机构大类=shop 的距截止日天数均值
4,block1_consultas,consultas_30d_shop_days_std,30,otorgante_group,shop,days_before_request,days_std,第一板块：近30天内 机构大类=shop 的距截止日天数标准差
5,block1_consultas,consultas_30d_shop_notnull_ratio,30,otorgante_group,shop_notnull,count,ratio,第一板块：近30天内 机构大类=shop_notnull 的占比（该类/总查询次数）
6,block1_consultas,consultas_30d_shop_daily_cnt_mean,30,otorgante_group,shop,daily_count,daily_cnt_mean,第一板块：近30天内 机构大类=shop 的每天平均次数（cnt/窗口天数）
7,block1_consultas,consultas_30d_shop_daily_cnt_std,30,otorgante_group,shop,daily_count,daily_cnt_std,第一板块：近30天内 机构大类=shop 的每天次数标准差（按窗口内每天计数序列）
8,block1_consultas,consultas_30d_mass_fin_assn_cnt,30,otorgante_group,mass_fin_assn,count,cnt,第一板块：近30天内 机构大类=mass_fin_assn 的次数
9,block1_consultas,consultas_30d_mass_fin_assn_ratio,30,otorgante_group,mass_fin_assn,count,ratio,第一板块：近30天内 机构大类=mass_fin_assn 的占比（该类/总查询次数）


In [82]:
# TODO（衍生报告：把评估结果写进一个 Excel，多 sheet）
# 大白话：你确认了“评估结果要写进一个 Excel 作为衍生报告”，所以这一格会把本 notebook 的评估输出落盘。
# 说明：
# - 如果 df_raw 里有 fpd7/approve_state：会额外产出分周坏率、IV、PSI、TopIV 分箱、分周IV。
# - 如果没有标签列：只输出特征画像（空值率/同一值率）与元信息。

import re  # 用于清洗 sheet 名
from pathlib import Path  # 用于拼接输出路径

import numpy as np  # 数值计算
import pandas as pd  # 表格计算

# 大白话：写报告开关（你已明确要写，所以默认 True）。
WRITE_REPORTS = False

# 大白话：输出文件路径（第一板块报告）。
REPORT_XLSX = Path("outputs") / "block1_eval_report.xlsx"

# 大白话：统一把 NaN 或 -999 当作空值（-999 是 A/C/D 的兜底值）。
SENTINEL_VALUE = -999

# 大白话：Excel 的 sheet 名最长 31 且不能重复，这里做一个安全命名。
def _safe_sheet_name(name: str, used: set[str]) -> str:
    base = re.sub(r"[^0-9a-zA-Z_]+", "_", str(name))[:31] or "sheet"  # 非法字符替换成下划线
    s = base  # 初始候选
    i = 1  # 冲突计数器
    while s in used:  # 如果重名就加后缀
        suf = f"_{i}"  # 后缀
        s = base[: max(0, 31 - len(suf))] + suf  # 保证仍<=31
        i += 1  # 递增
    used.add(s)  # 记录
    return s  # 返回安全 sheet 名

# 大白话：拿到原始数据与特征表（本 notebook 默认变量名是 df_raw / features）。
_raw = df_raw if "df_raw" in globals() else None  # 原始表
_feat = features if "features" in globals() else None  # 特征表

if _raw is None or _feat is None:
    raise RuntimeError("[REPORT] 未找到 df_raw 或 features，请先从头运行 notebook 生成它们。")

# 大白话：把 apply_id 作为索引，方便对齐。
_feat2 = _feat.copy()  # 复制，避免污染上游变量
if "apply_id" in _feat2.columns:
    _feat2 = _feat2.set_index("apply_id")  # 把 apply_id 设为索引

# 大白话：确保 request_time 是 datetime（用于分周）。
if "request_time" in _feat2.columns:
    _feat2["request_time"] = pd.to_datetime(_feat2["request_time"], errors="coerce")

# 大白话：X 是所有衍生特征列（不含 request_time）。
X = _feat2.drop(columns=["request_time"], errors="ignore")

# 大白话：空值掩码（NaN 或 -999）。
missing_mask = X.isna() | (X == SENTINEL_VALUE)

# 大白话：空值率（每列空值占比）。
na_rate = missing_mask.mean(axis=0)

# 大白话：同一值率（出现频率最高的值占比；空值统一当作同一个值来算）。
_same = {}
for c in X.columns:
    s = X[c].copy()
    s = s.mask(missing_mask[c], "__MISSING__")
    vc = s.value_counts(dropna=False, normalize=True)
    _same[c] = float(vc.iloc[0]) if len(vc) > 0 else 1.0
same_value_rate = pd.Series(_same).reindex(X.columns)

# 大白话：如果有标签 fpd7，就做更完整的评估（IV/分周坏率/PSI）。
has_fpd7 = "fpd7" in _raw.columns
has_state = "approve_state" in _raw.columns

# 大白话：对齐 y（坏=1 好=0）。
if has_fpd7:
    y = (
        _raw[["apply_id", "fpd7"]]
        .drop_duplicates("apply_id")
        .set_index("apply_id")["fpd7"]
        .pipe(pd.to_numeric, errors="coerce")
        .reindex(_feat2.index)
    )
    y01 = (y > 0).astype("int8")
else:
    y = None
    y01 = None

# 大白话：对齐 approve_state（cycle_pass / single_pass）。
if has_state:
    state = (
        _raw[["apply_id", "approve_state"]]
        .drop_duplicates("apply_id")
        .set_index("apply_id")["approve_state"]
        .fillna("")
        .astype(str)
        .str.lower()
        .reindex(_feat2.index)
        .fillna("")
    )
    mask_cycle = state.str.contains("cycle_pass", na=False)
    mask_single = state.str.contains("single_pass", na=False)
else:
    mask_cycle = pd.Series(False, index=_feat2.index)
    mask_single = pd.Series(False, index=_feat2.index)

mask_all = pd.Series(True, index=_feat2.index)

# 大白话：分周（周一 00:00:00）。
if "request_time" in _feat2.columns:
    t = _feat2["request_time"]
    week_start = (t - pd.to_timedelta(t.dt.weekday, unit="D")).dt.normalize()
else:
    week_start = pd.Series(pd.NaT, index=_feat2.index)

# ========== 可写入 Excel 的表集合 ==========
TABLES: dict[str, pd.DataFrame] = {}

# 1) 特征画像（空值率/同一值率）
profile_df = (
    pd.DataFrame({"feature": X.columns, "na_rate": na_rate.values, "same_value_rate": same_value_rate.values})
    .sort_values(["na_rate", "same_value_rate"], ascending=[False, False])
    .reset_index(drop=True)
)
TABLES["feature_profile"] = profile_df

# 2) 分周坏率（需要 y01）
if y01 is not None:
    def _weekly_bad_rate(mask: pd.Series, name: str) -> pd.DataFrame:
        idx = _feat2.index[mask]
        tmp = pd.DataFrame({"week": week_start.reindex(idx), "y": y01.reindex(idx)}).dropna(subset=["week", "y"])
        out = tmp.groupby("week", observed=False)["y"].agg(total_cnt="size", bad_cnt="sum").reset_index()
        out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
        out["sample"] = name
        return out

    weekly_bad = pd.concat(
        [
            _weekly_bad_rate(mask_all, "overall"),
            _weekly_bad_rate(mask_cycle, "cycle_pass"),
            _weekly_bad_rate(mask_single, "single_pass"),
        ],
        axis=0,
        ignore_index=True,
    ).sort_values(["week", "sample"]).reset_index(drop=True)

    TABLES["weekly_bad_rate"] = weekly_bad

# 3) IV / PSI / TopIV 分箱 / 分周 IV（需要 y01）
if y01 is not None:
    IV_Q = 10  # IV 分箱：分位数箱数
    BIN_N = 5  # 报告分箱：5箱

    def _compute_iv_one(x: pd.Series, ybin: pd.Series) -> float:
        dfv = pd.DataFrame({"x": x, "y": ybin}).dropna(subset=["y"])
        if dfv.shape[0] == 0:
            return float("nan")
        x_raw = dfv["x"]
        yb = dfv["y"].astype(int)
        miss = x_raw.isna() | (x_raw == SENTINEL_VALUE)
        x_non = x_raw[~miss]
        if x_non.nunique(dropna=True) <= 2:
            b = x_raw.astype(object).where(~miss, "MISSING")
        else:
            try:
                b_non = pd.qcut(x_non, q=IV_Q, duplicates="drop")
                b = pd.Series("MISSING", index=x_raw.index, dtype="object")
                b.loc[~miss] = b_non.astype(str)
            except Exception:
                b = x_raw.astype(object).where(~miss, "MISSING")
        grp = pd.DataFrame({"b": b, "y": yb}).groupby("b", observed=False)["y"].agg(["count", "sum"]).rename(columns={"sum": "bad"})
        grp["good"] = grp["count"] - grp["bad"]
        bt = grp["bad"].sum(); gt = grp["good"].sum()
        if bt == 0 or gt == 0:
            return 0.0
        k = grp.shape[0]
        grp["bad_dist"] = (grp["bad"] + 0.5) / (bt + 0.5 * k)
        grp["good_dist"] = (grp["good"] + 0.5) / (gt + 0.5 * k)
        woe = np.log(grp["bad_dist"] / grp["good_dist"])
        iv = ((grp["bad_dist"] - grp["good_dist"]) * woe).sum()
        return float(iv)

    # eligible：避免全空/近似常量
    eligible = (na_rate < 0.9) & (same_value_rate < 0.99)
    cols = X.columns[eligible].tolist()

    iv_overall = {c: _compute_iv_one(pd.to_numeric(X[c], errors="coerce"), y01) for c in cols}
    iv_overall_s = pd.Series(iv_overall, name="iv_overall")

    # 子样本 IV（如果没有 approve_state，会都是 NaN）
    iv_cycle_s = pd.Series({c: _compute_iv_one(pd.to_numeric(X[c], errors="coerce")[mask_cycle], y01[mask_cycle]) for c in cols}, name="iv_cycle_pass") if has_state and int(mask_cycle.sum())>0 else pd.Series(dtype="float64", name="iv_cycle_pass")
    iv_single_s = pd.Series({c: _compute_iv_one(pd.to_numeric(X[c], errors="coerce")[mask_single], y01[mask_single]) for c in cols}, name="iv_single_pass") if has_state and int(mask_single.sum())>0 else pd.Series(dtype="float64", name="iv_single_pass")

    # PSI：后两周 vs 第一周
    weeks = sorted([w for w in week_start.dropna().unique()])
    first_w = weeks[0] if len(weeks) >= 1 else None
    last_two = weeks[-2:] if len(weeks) >= 2 else weeks
    base_mask = (week_start == first_w) if first_w is not None else pd.Series(False, index=_feat2.index)
    comp_mask = week_start.isin(last_two) if len(last_two) > 0 else pd.Series(False, index=_feat2.index)

    PSI_Q = 10
    EPS = 1e-6

    def _psi_one(x: pd.Series) -> float:
        miss = x.isna() | (x == SENTINEL_VALUE)
        xb = x[base_mask]; xc = x[comp_mask]
        mb = miss[base_mask]; mc = miss[comp_mask]
        if xb.shape[0] == 0 or xc.shape[0] == 0:
            return float("nan")
        xb_non = xb[~mb]
        if xb_non.nunique(dropna=True) <= 2:
            bb = xb.astype(object).where(~mb, "MISSING")
            bc = xc.astype(object).where(~mc, "MISSING")
        else:
            try:
                _, edges = pd.qcut(xb_non, q=PSI_Q, retbins=True, duplicates="drop")
                edges = sorted(set(edges.tolist()))
                if len(edges) < 3:
                    bb = xb.astype(object).where(~mb, "MISSING")
                    bc = xc.astype(object).where(~mc, "MISSING")
                else:
                    bb_non = pd.cut(xb_non, bins=edges, include_lowest=True)
                    bc_non = pd.cut(xc[~mc], bins=edges, include_lowest=True)
                    bb = pd.Series("MISSING", index=xb.index, dtype="object"); bc = pd.Series("MISSING", index=xc.index, dtype="object")
                    bb.loc[~mb] = bb_non.astype(str); bc.loc[~mc] = bc_non.astype(str)
            except Exception:
                bb = xb.astype(object).where(~mb, "MISSING")
                bc = xc.astype(object).where(~mc, "MISSING")
        pb = bb.value_counts(normalize=True); pc = bc.value_counts(normalize=True)
        cats = list(pb.index.union(pc.index))
        psi = 0.0
        for k in cats:
            p = max(float(pb.get(k, 0.0)), EPS)
            q = max(float(pc.get(k, 0.0)), EPS)
            psi += (p - q) * float(np.log(p / q))
        return float(psi)

    # corr_fpd7
    fpd7_num = y.reindex(_feat2.index)

    iv_gt = iv_overall_s[iv_overall_s > 0.1].sort_values(ascending=False)
    rows = []
    for feat in iv_gt.index.tolist():
        x = pd.to_numeric(X[feat], errors="coerce")
        miss = x.isna() | (x == SENTINEL_VALUE)
        valid = (~miss) & fpd7_num.notna()
        corr = float(pd.Series(x[valid]).corr(fpd7_num[valid], method="pearson")) if int(valid.sum()) >= 3 else float("nan")
        rows.append(
            {
                "feature_name": feat,
                "iv": float(iv_gt.loc[feat]),
                "corr_fpd7": corr,
                "na_rate_na_or_-999": float(miss.mean()),
                "psi_last2w_vs_firstw": _psi_one(x) if (first_w is not None and len(last_two) > 0) else float("nan"),
                "iv_cycle_pass": float(iv_cycle_s.get(feat, float("nan"))),
                "iv_single_pass": float(iv_single_s.get(feat, float("nan"))),
            }
        )
    iv_report = pd.DataFrame(rows)
    for c in ["iv", "corr_fpd7", "na_rate_na_or_-999", "psi_last2w_vs_firstw", "iv_cycle_pass", "iv_single_pass"]:
        if c in iv_report.columns:
            iv_report[c] = pd.to_numeric(iv_report[c], errors="coerce").round(4)
    TABLES["iv_gt_0p1_report"] = iv_report

    # TopIV 特征（整体/cycle/single）
    top_overall = str(iv_overall_s.sort_values(ascending=False).index[0]) if len(iv_overall_s) > 0 else ""
    top_cycle = str(iv_cycle_s.sort_values(ascending=False).index[0]) if len(iv_cycle_s) > 0 else ""
    top_single = str(iv_single_s.sort_values(ascending=False).index[0]) if len(iv_single_s) > 0 else ""

    def _bin_report(feat: str, mask: pd.Series, method: str) -> pd.DataFrame:
        x = pd.to_numeric(X[feat], errors="coerce")[mask]
        yb = y01.reindex(X.index)[mask]
        miss = x.isna() | (x == SENTINEL_VALUE)
        x_non = x[~miss]
        if x_non.shape[0] == 0:
            lab = pd.Series("MISSING", index=x.index, dtype="object")
        else:
            if method == "qcut":
                r = x_non.rank(method="first")
                b_non = pd.qcut(r, q=BIN_N, duplicates="drop")
                lab = pd.Series("MISSING", index=x.index, dtype="object")
                lab.loc[~miss] = b_non.astype(str)
            else:
                b_non = pd.cut(x_non, bins=BIN_N, include_lowest=True)
                lab = pd.Series("MISSING", index=x.index, dtype="object")
                lab.loc[~miss] = b_non.astype(str)
        box = pd.DataFrame({"bin": lab, "y": yb}).dropna(subset=["y"])
        out = box.groupby("bin", observed=False)["y"].agg(total_cnt="size", bad_cnt="sum").reset_index()
        out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
        uniq = pd.DataFrame({"bin": lab, "x": x.where(~miss, np.nan)}).groupby("bin", observed=False)["x"].nunique(dropna=True).rename("unique_cnt").reset_index()
        out = out.merge(uniq, on="bin", how="left")
        # zlf update: 特征值为空时填充-999
        out["unique_cnt"] = out["unique_cnt"].fillna(0).astype(int)
        out.insert(0, "feature", feat)
        out.insert(1, "method", method)
        return out

    def _add_bins(tag: str, feat: str, mask: pd.Series):
        if not feat:
            return
        TABLES[f"bin_{tag}_qcut"] = _bin_report(feat, mask, "qcut")
        TABLES[f"bin_{tag}_cut"] = _bin_report(feat, mask, "cut")

    _add_bins("overall", top_overall, mask_all)
    _add_bins("cycle_pass", top_cycle, mask_cycle)
    _add_bins("single_pass", top_single, mask_single)

    # 分周IV（仅对 top 特征）
    def _weekly_iv(feat: str, mask: pd.Series) -> pd.DataFrame:
        idx = _feat2.index[mask]
        tmp = pd.DataFrame({"week": week_start.reindex(idx), "x": pd.to_numeric(X[feat], errors="coerce").reindex(idx), "y": y01.reindex(idx)}).dropna(subset=["week", "y"])
        out_rows = []
        for wk, g in tmp.groupby("week", observed=False):
            out_rows.append({"week": wk, "iv": _compute_iv_one(g["x"], g["y"]), "total_cnt": int(g.shape[0]), "bad_cnt": int(g["y"].sum())})
        out = pd.DataFrame(out_rows).sort_values("week")
        out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
        out["iv"] = pd.to_numeric(out["iv"], errors="coerce").round(4)
        return out

    for tag, feat in [("overall_top", top_overall), ("cycle_top", top_cycle), ("single_top", top_single)]:
        if not feat:
            continue
        TABLES[f"weekly_iv_{tag}_overall"] = _weekly_iv(feat, mask_all)
        TABLES[f"weekly_iv_{tag}_cycle_pass"] = _weekly_iv(feat, mask_cycle)
        TABLES[f"weekly_iv_{tag}_single_pass"] = _weekly_iv(feat, mask_single)

# 4) 元信息（永远写）
meta = pd.DataFrame(
    {
        "item": ["feature_cnt", "has_fpd7", "has_approve_state", "na_rate_ge_0.9_cnt", "same_value_rate_ge_0.99_cnt"],
        "value": [
            str(int(X.shape[1])),
            str(bool(has_fpd7)),
            str(bool(has_state)),
            str(int((na_rate >= 0.9).sum())),
            str(int((same_value_rate >= 0.99).sum())),
        ],
    }
)

# ========== 写 Excel（多 sheet） ==========
if WRITE_REPORTS:
    REPORT_XLSX.parent.mkdir(parents=True, exist_ok=True)
    used: set[str] = set()
    try:
        writer = pd.ExcelWriter(REPORT_XLSX, engine="openpyxl")
    except Exception:
        writer = pd.ExcelWriter(REPORT_XLSX)

    with writer:
        meta.to_excel(writer, sheet_name=_safe_sheet_name("meta", used), index=False)
        for name, df in TABLES.items():
            if df is None:
                df = pd.DataFrame()
            df.to_excel(writer, sheet_name=_safe_sheet_name(name, used), index=False)

    print("[WRITE_EXCEL] block1 eval report saved:", str(REPORT_XLSX.resolve()))

# =========================
# 额外：生成 quality 检测 Excel（每个特征一行）
# =========================
# 大白话：你要求每个板块 notebook 自己也能落一个“quality 质量表 Excel”，这里按同一套口径生成：
# - 空值=NaN/NaT 或 -999
# - y=fpd7>0 视为坏=1
# - PSI=后两周 vs 第一周（按 request_time 的周一对齐）
# - 同时计算 cycle_pass / single_pass 子样本 IV
# 注意：这一格默认不写出文件；你确认要写再打开开关。

WRITE_QUALITY_EXCEL = False  # 大白话：True 才会写出 quality Excel
QUALITY_XLSX = Path("outputs") / "block1_feature_quality.xlsx"  # 大白话：第一板块 quality 输出文件名

if WRITE_QUALITY_EXCEL:
    import scripts.build_all_blocks_feature_quality_excel as q  # 大白话：复用我们已验证过的 quality 计算逻辑（IV/PSI/相关性等）

    # 大白话：准备 base（y / approve_state / request_time）。
    _base = df_raw[["apply_id", "request_time", "fpd7", "approve_state"]].copy()  # 大白话：取必要列
    _base["apply_id"] = _base["apply_id"].astype(str)  # 大白话：统一 id 类型
    _base = _base.drop_duplicates("apply_id").set_index("apply_id")  # 大白话：一人一行

    _y = (pd.to_numeric(_base["fpd7"], errors="coerce") > 0).astype(int)  # 大白话：Y=fpd7>0
    _state = _base["approve_state"].fillna("").astype(str).str.lower()  # 大白话：标准化状态字符串
    _is_cycle = _state.str.contains("cycle_pass", na=False)  # 大白话：cycle_pass 子样本
    _is_single = _state.str.contains("single_pass", na=False)  # 大白话：single_pass 子样本

    # 大白话：按 request_time 算周（周一 00:00:00）。
    _rt = pd.to_datetime(_base["request_time"], errors="coerce")
    _week_start = (_rt - pd.to_timedelta(_rt.dt.weekday, unit="D")).dt.normalize()
    _weeks = sorted([w for w in _week_start.dropna().unique()])
    _first_w = _weeks[0] if len(_weeks) >= 1 else None
    _last_two = _weeks[-2:] if len(_weeks) >= 2 else _weeks
    _base_mask = (_week_start == _first_w) if _first_w is not None else pd.Series(False, index=_week_start.index)
    _comp_mask = _week_start.isin(_last_two) if len(_last_two) > 0 else pd.Series(False, index=_week_start.index)

    # 大白话：准备 features（只取衍生列，不含 request_time）。
    _feat = features.copy()  # 大白话：features index=apply_id
    if "apply_id" in _feat.columns:
        _feat = _feat.set_index("apply_id")
    _feat.index = _feat.index.astype(str)
    _X = _feat.drop(columns=["request_time"], errors="ignore")

    # 大白话：准备特征字典（feature_name 不带前后缀）。
    _dict = feature_dict_3col.copy()
    _dict["feature_name"] = _dict["feature_name"].astype(str)
    _dict = _dict.drop_duplicates("feature_name").set_index("feature_name")

    _rows = []
    _total = int(_X.shape[0])

    for _col in _X.columns:
        _col_name = str(_col)
        _base_name = q._strip_feature_name(_col_name)  # 大白话：cdc_{base}_v2 -> {base}

        _x = pd.to_numeric(_X[_col], errors="coerce")
        _miss = _x.isna() | (_x == q.SENTINEL)
        _na_cnt = int(_miss.sum())
        _na_rate = float(_na_cnt / _total) if _total else float("nan")
        _non = _x[~_miss]
        _nunique = int(_non.nunique(dropna=True))

        _cn = str(_dict["cn_desc"].get(_base_name, "")) if _base_name in _dict.index else ""
        _logic = str(_dict["logic_desc"].get(_base_name, "")) if _base_name in _dict.index else ""

        _corr = float("nan")
        _valid = (~_miss) & _y.notna()
        if int(_valid.sum()) >= 3:
            _corr = q._corr_pearson(_x[_valid].to_numpy(dtype="float64", copy=False), _y[_valid].to_numpy(dtype="float64", copy=False))

        _iv = q._iv_one(_x, _y)
        _psi = q._psi_one(_x, _base_mask, _comp_mask)
        _iv_c = q._iv_one(_x[_is_cycle], _y[_is_cycle])
        _iv_s = q._iv_one(_x[_is_single], _y[_is_single])

        _rows.append(
            {
                "feature_name": _base_name,
                "cn_desc": _cn,
                "logic_desc": _logic,
                "na_cnt": _na_cnt,
                "total_cnt": _total,
                "na_rate": round(_na_rate, 6),
                "nunique": _nunique,
                "corr_y": None if pd.isna(_corr) else round(float(_corr), 6),
                "iv": None if pd.isna(_iv) else round(float(_iv), 6),
                "psi_last2w_vs_firstw": None if pd.isna(_psi) else round(float(_psi), 6),
                "iv_cycle_pass": None if pd.isna(_iv_c) else round(float(_iv_c), 6),
                "iv_single_pass": None if pd.isna(_iv_s) else round(float(_iv_s), 6),
                "notebook": "第一板块衍生.ipynb",
            }
        )

    _quality = pd.DataFrame(_rows)
    QUALITY_XLSX.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(QUALITY_XLSX) as _w:
        _quality.to_excel(_w, sheet_name="report", index=False)
    print("[WRITE_EXCEL] block1 feature quality saved:", str(QUALITY_XLSX.resolve()))



FileNotFoundError: [Errno 2] No such file or directory: 'CDC/outputs/cdc1_features_batch001_1-500.csv'